# Format sampling results

In [1]:
import re
import sys
from ast import literal_eval

import pandas as pd
from rdkit import Chem

sys.path.append('..')

from modules.core.features.molecule_filter import MoleculeFilter

filter = MoleculeFilter()

In [ ]:
df = pd.read_csv('../data/sampling/filtering_comparison.csv')
df["smiles_after_filtering"] = df["smiles_after_filtering"].apply(literal_eval)
df.shape

In [ ]:
df.head()

In [2]:
experts1 = pd.read_csv('../data/processed_all_custom_features/data_experts_1.csv')
experts2 = pd.read_csv('../data/raw/data_experts2.csv')
experts3 = pd.read_csv('../data/raw/data_experts3.csv')

# canonicalize smiles for all experts
experts1["smiles"] = experts1["smiles"].apply(lambda x: Chem.CanonSmiles(x))
experts2["smiles"] = experts2["smiles"].apply(lambda x: Chem.CanonSmiles(x))
experts3["smiles"] = experts3["smiles"].apply(lambda x: Chem.CanonSmiles(x))

In [3]:
df_experts = pd.concat([experts1, experts2, experts3], ignore_index=True)["smiles"]
print(df_experts.shape)
df_experts = set(df_experts.to_list())
len(df_experts)

(76,)


75

In [ ]:
def parse_filenames(df, column_name):
    """
    Parse a column of filenames into separate features.
    
    Args:
        df (pd.DataFrame): Input DataFrame.
        column_name (str): Name of the column to parse.
    
    Returns:
        pd.DataFrame: DataFrame with new feature columns.
    """
    # Define patterns to extract features
    def parse_row(row):
        pattern = {
            "model_name": r"(bionemo_[a-z]+|reinvent|filtered|symmetrical)",  # Model name
            "seed": r"(data_experts_\d+)",  # Seed information
            "num_samples": r"num_samples_(\d+)|(\d+smiles)",  # Number of samples (bionemo or reinvent style)
            "sampling_method": r"sampling_method_([\w-]+)",  # Sampling method
            "scaled_radius": r"scaled_radius_(\d+(\.\d+)?)",  # Scaled radius
            "chunks": r"(\d+x\d+)_chunks",  # Chunk info
            "beam_size": r"beam_size_(\d+)",  # Beam size
            "beam_alpha": r"beam_alpha_(\d+(\.\d+)?)",  # Beam alpha
            "top_k": r"top_k_(\d+)",  # Top-k value
            "top_p": r"top_p_(\d+(\.\d+)?)",  # Top-p value
            "temperature": r"temperature_(\d+(\.\d+)?)"  # Temperature
        }
        parsed_data = {}
        for key, regex in pattern.items():
            match = re.search(regex, row)
            if key == "model_name":
                # Remove "bionemo_" prefix if it's a bionemo model
                parsed_data[key] = match.group(1).replace("bionemo_", "") if match and "bionemo_" in match.group(1) else match.group(1)
                # if name is filtered, then it is a reinvent
                if parsed_data[key] == "filtered" or parsed_data[key] == "symmetrical":
                    parsed_data[key] = "reinvent"
            elif key == "seed":
                # Extract seed (data_experts_x)
                parsed_data[key] = match.group(1) if match else None
            elif key == "num_samples":
                if match:  # Check if a match is found
                    num_match = match.group(1) or match.group(2)
                    parsed_data[key] = re.sub(r"smiles", "", num_match) if num_match else None
                else:
                    parsed_data[key] = None if not match else match.group(1)
            else:
                parsed_data[key] = match.group(1) if match else None
        return parsed_data

    # Apply parsing logic to each row
    parsed_data = df[column_name].apply(parse_row).apply(pd.Series)
    
    # Concatenate original DataFrame with parsed columns
    df = pd.concat([df, parsed_data], axis=1)
    return df


df = parse_filenames(df, "filenames")
# remove duplicate columns
df = df.loc[:, ~df.columns.duplicated()]
df.to_csv('../data/sampling/filtering_comparison_parsed.csv', index=False)
# # group by model and mean over the number of samples
df[["model_name", "num_total_molecules", "num_filtered_molecules"]].groupby("model_name").mean()

In [ ]:
# group by model name and aggregate molecules after filtering
res_df = df.groupby("model_name")["smiles_after_filtering"].agg(lambda x: [item for sublist in x for item in sublist])
# apply set to remove duplicates
res_df = res_df.apply(set)


print(res_df.apply(len))

# exclude experts from the set
res_df = res_df.apply(lambda x: x - df_experts)

print(res_df.apply(len))

In [ ]:
res_df.to_csv('../data/sampling/xlsx_input.csv', index=True)

In [ ]:
# TODO: execute script to generate excel

## Experts data which passed the filters
Go to root as MoleculeFilter has paths relative to root 

In [4]:
cd .. 

/home/witoldt/repositories/battery-rangers


/home/witoldt/repositories/battery-rangers/.venv/lib/python3.12/site-packages/IPython/core/magics/osm.py:417: UserWarning: This is now an optional IPython functionality, setting dhist requires you to install the `pickleshare` library.
  self.shell.db['dhist'] = compress_dhist(dhist)[-100:]


In [26]:
experts_dict = {
    "experts1": experts1.copy(),
    "experts2": experts2.copy(),
    "experts3": experts3.copy()
}

for k, v in experts_dict.items():
    
    filtered_results = filter.apply_against_all_filters(v["smiles"].to_list())
    experts_dict[k] = pd.DataFrame(filtered_results).transpose()

INFO:modules.core.features.molecule_filter:Checking filters for 24 molecules.
INFO:modules.core.features.molecule_filter:Number of molecules that could be converted to RDKit Mol objects: 24
INFO:modules.core.features.molecule_filter:Molecule C#Cc1ccc(C#N)cc1 passed filter SymmetryFilter in 0.00 s.
INFO:modules.core.features.molecule_filter:Molecule C#Cc1ccc(C#N)cc1 failed filter CNTripleBondsFilter in 0.00 s.
INFO:modules.core.features.molecule_filter:Molecule C#Cc1ccc(C#N)cc1 passed filter PointGroupSymmetryFilter in 0.02 s.
Calculating flatness: 100%|██████████| 1/1 [00:00<00:00, 27.26it/s]
INFO:modules.core.features.molecule_filter:Molecule C#Cc1ccc(C#N)cc1 passed filter FlatnessFilter in 0.04 s.
INFO:modules.core.features.molecule_filter:Molecule C#Cc1ccc(C#N)cc1 passed filter StericHindranceFilter in 0.01 s.
INFO:modules.core.features.molecule_filter:Molecule C#Cc1ccc(C#N)cc1 passed filter XYZPatternFilter in 0.00 s.
INFO:modules.core.features.molecule_filter:Molecule CC(C)c1ccc(-

In [31]:
import os
import shutil

import pandas as pd
from openpyxl import Workbook
from openpyxl.drawing.image import Image
from openpyxl.worksheet.table import Table, TableStyleInfo # Import Table and TableStyleInfo
from rdkit import Chem
from rdkit.Chem import Draw


def generate_excel_from_dict(
    model_data: dict, output_file: str, output_folder: str = "temp_images"
):
    """
    Generates an Excel file with SMILES structures, images, and filter pass/fail
    columns for each filter, directly from a dictionary input.

    Args:
        model_data (dict): Dictionary of models and their SMILES DataFrames,
            where DataFrames MUST have SMILES as index and filter columns
            with boolean results.
        output_file (str): Path to save the output Excel file.
        output_folder (str): Path to the folder for temporary images.
            Defaults to 'temp_images'.

    Returns:
        None
    """
    # Create a new Excel workbook
    wb = Workbook()

    # Create the output folder for temporary images
    os.makedirs(output_folder, exist_ok=True)

    for model_name, df in model_data.items():
        # Create a new sheet for each model
        ws = wb.create_sheet(title=model_name)

        # Add headers
        headers = ["SMILES", "Structure", "Grade (0-5)"]
        filter_columns = [col for col in df.columns if col != 'smiles'] # Assuming 'smiles' column is explicitly present or index is used.
        filter_names = [f"Passed {col}" for col in filter_columns]
        headers.extend(filter_names) # Add filter pass/fail headers
        header_row = 1
        for col_num, header in enumerate(headers, start=1):
            ws.cell(row=header_row, column=col_num, value=header)

        # Set default column widths
        ws.column_dimensions["A"].width = 30  # SMILES column
        ws.column_dimensions["B"].width = 50  # Image column
        ws.column_dimensions["C"].width = 20  # Grade column
        start_filter_col = 4
        for i in range(len(filter_names)):
            ws.column_dimensions[chr(ord('D') + i)].width = 15  # Passed Filter columns, start from D

        # Set default row height
        for row in range(2, len(df) + 2):  # Adjust for data rows
            ws.row_dimensions[row].height = 250

        # Process each SMILES in the DataFrame
        for i, row in enumerate(df.iterrows(), start=2):
            smiles = df.index[i-2] # SMILES is now index
            data = df.iloc[i-2] # Access data via iloc

            # Parse the molecule
            mol = Chem.MolFromSmiles(smiles)

            # Add SMILES to the Excel
            ws.cell(row=i, column=1, value=smiles)

            img_path = os.path.join(output_folder, f"{model_name}_mol_{i}.png")
            if mol: # Only draw if Mol object is valid
                Draw.MolToFile(mol, img_path)
                # Add the image to the Excel
                img = Image(img_path)
                ws.add_image(img, f"B{i}")
            else:
                ws.cell(row=i, column=2, value="Invalid SMILES") # Indicate invalid SMILES if Mol is None

            # Add grading instructions
            ws.cell(row=i, column=3, value="Enter 0-5")

            # Add filter pass/fail results dynamically
            filter_result_col = start_filter_col
            for filter_col_name in filter_columns:
                passed_filter = data[filter_col_name] # Get boolean value directly from DataFrame
                ws.cell(row=i, column=filter_result_col, value="True" if passed_filter else "False")
                filter_result_col += 1
        # Define table range (A1 to last column and last row)
        last_col_letter = chr(ord('A') + len(headers) - 1) # Calculate last column letter
        last_row_num = len(df) + 1 # Header row + data rows
        table_range = f"A1:{last_col_letter}{last_row_num}"

        # Create table style
        style = TableStyleInfo(name="TableStyleMedium9", showFirstColumn=False,
                               showLastColumn=False, showRowStripes=True, showColumnStripes=False)

        # Create Table object
        tab = Table(displayName=f"Table_{model_name}", ref=table_range, tableStyleInfo=style)

        # Add the table to the worksheet
        ws.add_table(tab)


    # Remove the default sheet
    if "Sheet" in wb.sheetnames:
        del wb["Sheet"]

    # Save the Excel file
    wb.save(output_file)
    print(f"Excel file saved as {output_file}.")
    if os.path.exists(output_folder):
         shutil.rmtree(output_folder)
         print(f"Temporary folder '{output_folder}' has been deleted.")




generate_excel_from_dict(experts_dict, "experts_filters_new.xlsx")

Excel file saved as experts_filters_new.xlsx.
Temporary folder 'temp_images' has been deleted.
